# 🔬 Lifelong Image Retrieval using DwoPP & Episodic Metric Learning (Kaggle 2-GPU)
Notebook này hướng dẫn chi tiết cách chạy huấn luyện tăng trưởng và đánh giá hệ thống truy xuất ảnh suốt đời (**Lifelong Image Retrieval**) cho cả **4 Tasks** tuần tự trên Kaggle.

### ⚙️ Thiết kế mô hình & hàm Loss mới (DwoPP):
1. **Retrieval Projection Head**: Một lớp chiếu Conv2D cục bộ được tích hợp vào Decoupled Head của YOLO-World để ánh xạ đặc trưng vùng về 256 chiều.
2. **Episodic Hard-mining Metric Loss ($L_{eps}$)**: Triplet Loss áp dụng Batch-Hard mining để tối ưu hóa khoảng cách Euclidean của các mẫu dương cục bộ.
3. **Distillation without Positive Pairs ($L_{DwoPP}$)**: Hàm loss chưng cất tri thức từ mô hình nhiệm vụ trước nhưng loại bỏ hoàn toàn lớp tích cực (positive class) khỏi phân phối xác suất nhằm bảo toàn không gian metric mà không bị quên lãng thảm họa.
4. **Text Projection Layer**: Ánh xạ class embeddings văn bản từ 512 chiều về 256 chiều để đối sánh trực tiếp với đặc trưng vùng ảnh.

### ⚙️ Hỗ trợ Checkpoint Pre-trained:
* Nếu bạn đã pre-train trước các checkpoint phát hiện đối tượng (Object Detection) gốc của mô hình, bạn có thể cấu hình để nạp thẳng các checkpoint đó tại mỗi Task để học căn chỉnh không gian metric một cách nhanh chóng.

### ⚠️ Yêu cầu trước khi chạy:
1. Hãy chắc chắn rằng bạn đã kích hoạt **GPU T4 x2** trong phần settings của Kaggle (*Accelerator -> GPU T4 x2*).
2. Bật kết nối internet cho notebook (*Internet on*).

## 🛠️ Bước 1: Clone Repository & Submodules

In [ ]:
import os
repo_url = "https://github.com/nta2112/OW_OVD-An-custom.git"
working_dir = "/kaggle/working/OW_OVD"

if not os.path.exists(working_dir):
    print("-> Đang clone repository từ GitHub...")
    !git clone {repo_url} {working_dir}
else:
    print("-> Repository đã tồn tại. Đang tiến hành cập nhật (git pull)...")
    %cd {working_dir}
    !git pull

%cd {working_dir}

# Tải mmyolo vào thư mục third_party nếu chưa có
if not os.path.exists("third_party/mmyolo"):
    print("-> Đang tải submodule mmyolo...")
    !git clone https://github.com/open-mmlab/mmyolo.git third_party/mmyolo
else:
    print("-> Submodule mmyolo đã có sẵn.")

## 📦 Bước 2: Cài đặt Dependencies & Vá lỗi MMCV

In [ ]:
print("-> 1. Thiết lập phiên bản PyTorch & Torchvision...")
!pip install -q torch==2.4.0+cu121 torchvision==0.19.0+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

print("\n-> 2. Cài đặt MMCV từ wheel index...")
!pip install -q mmcv -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html

print("\n-> 3. Cài đặt các thư viện bổ trợ...")
!pip install -q matplotlib pycocotools terminaltables mmengine prettytable wcwidth open_clip_torch transformers

print("\n-> 4. Cài đặt MMDetection...")
!pip install -q "mmdet>=3.1.0" --no-deps

print("\n-> 5. Cài đặt MMYOLO từ source...")
!pip install -q --no-build-isolation --no-deps third_party/mmyolo

print("\n-> 6. Vá lỗi kiểm tra phiên bản MMCV vật lý trên đĩa cứng...")
import site
import os
import glob
import shutil

def patch_file(file_path):
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        new_content = content
        for old_ver in ["'2.1.0'", "'2.2.0'", '"2.1.0"', '"2.2.0"']:
            new_content = new_content.replace(f"mmcv_maximum_version = {old_ver}", "mmcv_maximum_version = '2.3.0'")
        if new_content != content:
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(new_content)
            print(f"  [Vá lỗi] Đã cập nhật file: {file_path}")

def clear_pycache(root_dir):
    if not os.path.exists(root_dir):
        return
    for root, dirs, files in os.walk(root_dir):
        for d in dirs:
            if d == "__pycache__":
                pycache_path = os.path.join(root, d)
                try:
                    shutil.rmtree(pycache_path)
                except Exception:
                    pass

site_dirs = site.getsitepackages()
for s_dir in site_dirs:
    for pkg in ["mmdet", "mmyolo"]:
        pkg_dir = os.path.join(s_dir, pkg)
        patch_file(os.path.join(pkg_dir, "__init__.py"))
        clear_pycache(pkg_dir)

for init_file in glob.glob("**/mmyolo/__init__.py", recursive=True):
    patch_file(init_file)
    clear_pycache(os.path.dirname(init_file))
for init_file in glob.glob("**/mmdet/__init__.py", recursive=True):
    patch_file(init_file)
    clear_pycache(os.path.dirname(init_file))

paths_to_glob = [
    "/opt/conda/lib/python*/site-packages/mmdet/__init__.py",
    "/opt/conda/lib/python*/site-packages/mmyolo/__init__.py",
    "/usr/local/lib/python*/dist-packages/mmdet/__init__.py",
    "/usr/local/lib/python*/dist-packages/mmyolo/__init__.py"
]
for path_pattern in paths_to_glob:
    for init_file in glob.glob(path_pattern):
        patch_file(init_file)
        clear_pycache(os.path.dirname(init_file))

print("\n-> 7. Kiểm tra import tất cả các package...")
import torch
import mmcv

real_mmcv_version = mmcv.__version__
mmcv.__version__ = '2.0.1'

import mmdet
import mmyolo
mmcv.__version__ = real_mmcv_version

print(f"  - torch: {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"  - mmcv: {mmcv.__version__}")
print(f"  - mmdet: {mmdet.__version__}")
print(f"  - mmyolo: {mmyolo.__version__}")
print("====== Khởi tạo môi trường hoàn tất! ======")

## 🗂️ Bước 3: Định vị Dataset & Sinh Đặc trưng nhãn bằng CLIP

In [ ]:
import json
import torch
import numpy as np
import os
import glob
from transformers import AutoTokenizer, CLIPTextModelWithProjection

os.makedirs('pretrained_models', exist_ok=True)
os.makedirs('data/IP102', exist_ok=True)
os.makedirs('data/texts/IP102', exist_ok=True)

weights_path = 'pretrained_models/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth'
if not os.path.exists(weights_path):
    print("-> Đang tải pretrained weights của YOLO-World...")
    !wget -O {weights_path} https://huggingface.co/wondervictor/YOLO-World/resolve/main/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth

dataset_root = None
for path in [
    '/kaggle/input/datasets/nta212/ip102-for-object-detection',
    '/kaggle/input/ip102-for-object-detection',
    'data/IP102',
    '.'
]:
    if os.path.exists(os.path.join(path, 'train.json')):
        dataset_root = path
        break
if dataset_root is None:
    paths = glob.glob('/kaggle/input/**/train.json', recursive=True)
    if paths:
        dataset_root = os.path.dirname(paths[0])

print(f"-> Thư mục Dataset IP102: {dataset_root}")
class_names = [str(i) for i in range(102)]

class_texts = [[name] for name in class_names]
with open('data/texts/IP102/class_texts.json', 'w') as f:
    json.dump(class_texts, f)

print("-> Đang sinh text embeddings bằng CLIP...")
model_name = 'openai/clip-vit-base-patch32'
tokenizer = AutoTokenizer.from_pretrained(model_name)
clip_model = CLIPTextModelWithProjection.from_pretrained(model_name, use_safetensors=True)
clip_model.eval()

embeddings = []
with torch.no_grad():
    for name in class_names:
        inputs = tokenizer(name, padding=True, return_tensors="pt")
        outputs = clip_model(**inputs)
        embed = outputs.text_embeds[0].cpu().numpy()
        embed = embed / np.linalg.norm(embed)
        embeddings.append(embed)

np.save('data/IP102/ip102_gt_embeddings.npy', np.array(embeddings))

num_att = len(class_names) * 25
torch.save({
    'att_embedding': torch.zeros(num_att, 512),
    'att_text': [f"att_{i}" for i in range(num_att)]
}, 'data/IP102/task_att_1_embeddings.pth')

thrs = [0.55]
pos_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
neg_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
torch.save({
    'positive_distributions': pos_dist,
    'negative_distributions': neg_dist
}, 'data/IP102/mowod_distribution_sim1.pth')
print("====== Khởi tạo và sinh đặc trưng nhãn hoàn tất! ======")

## ⚙️ Bước 4: Khai báo Checkpoint Phát hiện có sẵn (Pre-trained Detection Checkpoints)
Nếu bạn đã huấn luyện trước các checkpoint phát hiện đối tượng gốc và muốn dùng checkpoint đó làm khởi tạo để chỉ tập trung tối ưu không gian metric truy xuất:
* Hãy điền đường dẫn checkpoint vào biến `PRETRAINED_DET_CHECKPOINTS` bên dưới.
* Nếu không có, hãy giữ giá trị `None` để hệ thống tự động học nối tiếp từ đầu.

In [ ]:
PRETRAINED_DET_CHECKPOINTS = {
    "task_1": "/kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3/best_coco_Current class AP50_epoch_5.pth",
    "task_2": "/kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3/ip102_t2.pth",
    "task_3": "/kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3/ip102_t3.pth",
    "task_4": "/kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3/ip102_t4.pth"
}

print("-> Đã khai báo cấu hình checkpoints ban đầu.")

## 🚀 Bước 5: Huấn luyện & Đánh giá Nhiệm vụ 1 (Task 1 - 7 Lớp đầu)
Huấn luyện khớp không gian metric của 7 lớp đầu tiên.

In [ ]:
import subprocess
import os

config_path = "NewRetrieval_02/ip102_t1_retrieval.py"
checkpoint_save_dir = "work_dirs/ip102_t1_retrieval"

init_checkpoint = PRETRAINED_DET_CHECKPOINTS["task_1"]
if init_checkpoint is None:
    init_checkpoint = "pretrained_models/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth"

print(f"-> Bắt đầu huấn luyện Task 1 từ checkpoint: {init_checkpoint}")

# Sử dụng subprocess.run và danh sách tham số để tránh hoàn toàn lỗi dấu ngoặc kép hoặc khoảng trắng của shell
os.environ["PYTHONPATH"] = "."
cmd = [
    "torchrun",
    "--nproc_per_node=2",
    "--master_port=29500",
    "third_party/mmyolo/tools/train.py",
    config_path,
    "--launcher", "pytorch",
    "--cfg-options", f"load_from={init_checkpoint}"
]
subprocess.run(cmd, check=True)

In [ ]:
import subprocess
import os

print("-> Đang thực hiện đánh giá suốt đời sau Task 1...")
best_checkpoint = "work_dirs/ip102_t1_retrieval/best_coco_Current class AP50_epoch_1.pth"

os.environ["PYTHONPATH"] = "."
cmd = [
    "python",
    "NewRetrieval_02/evaluate_retrieval_lifelong.py",
    "--config", "NewRetrieval_02/ip102_t1_retrieval.py",
    "--checkpoint", best_checkpoint,
    "--dataset-root", dataset_root,
    "--current-task", "1",
    "--detector-retrieval",
    "--query-cache", "query_cache_t1.pkl",
    "--gallery-cache", "gallery_cache_t1.pkl",
    "--output-report", "retrieval_lifelong_report_t1.md",
    "--history-file", "history_metrics.json"
]
subprocess.run(cmd, check=True)

## 🚀 Bước 6: Huấn luyện & Đánh giá Nhiệm vụ 2 (Task 2 - Thêm 6 lớp mới là 13 Lớp)
Nạp checkpoint học được từ Task 1 (hoặc checkpoint pretrain của Task 2 nếu khai báo) để tiếp tục huấn luyện.

In [ ]:
import subprocess
import os

config_path = "NewRetrieval_02/ip102_t2_retrieval.py"

init_checkpoint = PRETRAINED_DET_CHECKPOINTS["task_2"]
if init_checkpoint is None:
    init_checkpoint = "work_dirs/ip102_t1_retrieval/best_coco_Current class AP50_epoch_1.pth"

print(f"-> Bắt đầu huấn luyện Task 2 từ checkpoint: {init_checkpoint}")

os.environ["PYTHONPATH"] = "."
cmd = [
    "torchrun",
    "--nproc_per_node=2",
    "--master_port=29501",
    "third_party/mmyolo/tools/train.py",
    config_path,
    "--launcher", "pytorch",
    "--cfg-options", f"load_from={init_checkpoint}"
]
subprocess.run(cmd, check=True)

In [ ]:
import subprocess
import os

print("-> Đang thực hiện đánh giá suốt đời sau Task 2...")
best_checkpoint = "work_dirs/ip102_t2_retrieval/best_coco_Current class AP50_epoch_1.pth"

os.environ["PYTHONPATH"] = "."
cmd = [
    "python",
    "NewRetrieval_02/evaluate_retrieval_lifelong.py",
    "--config", "NewRetrieval_02/ip102_t2_retrieval.py",
    "--checkpoint", best_checkpoint,
    "--dataset-root", dataset_root,
    "--current-task", "2",
    "--detector-retrieval",
    "--query-cache", "query_cache_t2.pkl",
    "--gallery-cache", "gallery_cache_t2.pkl",
    "--output-report", "retrieval_lifelong_report_t2.md",
    "--history-file", "history_metrics.json"
]
subprocess.run(cmd, check=True)

## 🚀 Bước 7: Huấn luyện & Đánh giá Nhiệm vụ 3 (Task 3 - Thêm 6 lớp mới là 19 Lớp)

In [ ]:
import subprocess
import os

config_path = "NewRetrieval_02/ip102_t3_retrieval.py"

init_checkpoint = PRETRAINED_DET_CHECKPOINTS["task_3"]
if init_checkpoint is None:
    init_checkpoint = "work_dirs/ip102_t2_retrieval/best_coco_Current class AP50_epoch_1.pth"

print(f"-> Bắt đầu huấn luyện Task 3 từ checkpoint: {init_checkpoint}")

os.environ["PYTHONPATH"] = "."
cmd = [
    "torchrun",
    "--nproc_per_node=2",
    "--master_port=29502",
    "third_party/mmyolo/tools/train.py",
    config_path,
    "--launcher", "pytorch",
    "--cfg-options", f"load_from={init_checkpoint}"
]
subprocess.run(cmd, check=True)

In [ ]:
import subprocess
import os

print("-> Đang thực hiện đánh giá suốt đời sau Task 3...")
best_checkpoint = "work_dirs/ip102_t3_retrieval/best_coco_Current class AP50_epoch_1.pth"

os.environ["PYTHONPATH"] = "."
cmd = [
    "python",
    "NewRetrieval_02/evaluate_retrieval_lifelong.py",
    "--config", "NewRetrieval_02/ip102_t3_retrieval.py",
    "--checkpoint", best_checkpoint,
    "--dataset-root", dataset_root,
    "--current-task", "3",
    "--detector-retrieval",
    "--query-cache", "query_cache_t3.pkl",
    "--gallery-cache", "gallery_cache_t3.pkl",
    "--output-report", "retrieval_lifelong_report_t3.md",
    "--history-file", "history_metrics.json"
]
subprocess.run(cmd, check=True)

## 🚀 Bước 8: Huấn luyện & Đánh giá Nhiệm vụ 4 (Task 4 - Thêm 5 lớp cuối là 25 Lớp)

In [ ]:
import subprocess
import os

config_path = "NewRetrieval_02/ip102_t4_retrieval.py"

init_checkpoint = PRETRAINED_DET_CHECKPOINTS["task_4"]
if init_checkpoint is None:
    init_checkpoint = "work_dirs/ip102_t3_retrieval/best_coco_Current class AP50_epoch_1.pth"

print(f"-> Bắt đầu huấn luyện Task 4 từ checkpoint: {init_checkpoint}")

os.environ["PYTHONPATH"] = "."
cmd = [
    "torchrun",
    "--nproc_per_node=2",
    "--master_port=29503",
    "third_party/mmyolo/tools/train.py",
    config_path,
    "--launcher", "pytorch",
    "--cfg-options", f"load_from={init_checkpoint}"
]
subprocess.run(cmd, check=True)

In [ ]:
import subprocess
import os

print("-> Đang thực hiện đánh giá suốt đời sau Task 4...")
best_checkpoint = "work_dirs/ip102_t4_retrieval/best_coco_Current class AP50_epoch_1.pth"

os.environ["PYTHONPATH"] = "."
cmd = [
    "python",
    "NewRetrieval_02/evaluate_retrieval_lifelong.py",
    "--config", "NewRetrieval_02/ip102_t4_retrieval.py",
    "--checkpoint", best_checkpoint,
    "--dataset-root", dataset_root,
    "--current-task", "4",
    "--detector-retrieval",
    "--query-cache", "query_cache_t4.pkl",
    "--gallery-cache", "gallery_cache_t4.pkl",
    "--output-report", "retrieval_lifelong_report_t4.md",
    "--history-file", "history_metrics.json"
]
subprocess.run(cmd, check=True)

## 📊 Bước 9: Tổng hợp và hiển thị Ma trận học trọn đời (Lifelong Performance Matrix)
Hiển thị chi tiết bảng so sánh chất lượng truy xuất qua các pha huấn luyện để theo dõi mức độ ổn định của thuật toán chưng cất DwoPP.

In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown

if os.path.exists("history_metrics.json"):
    with open("history_metrics.json", "r") as f:
        history = json.load(f)
        
    rows = []
    for stage, metrics in sorted(history.items()):
        rows.append({
            "Giai đoạn Đánh giá": stage.upper().replace("_", " "),
            "T1 mAP (7 lớp đầu)": f"{metrics.get('T1', {}).get('mAP', 0.0):.4f}",
            "T2 mAP (lớp 8-13)": f"{metrics.get('T2', {}).get('mAP', 0.0):.4f}",
            "T3 mAP (lớp 14-19)": f"{metrics.get('T3', {}).get('mAP', 0.0):.4f}",
            "T4 mAP (lớp 20-25)": f"{metrics.get('T4', {}).get('mAP', 0.0):.4f}",
        })
        
    df = pd.DataFrame(rows)
    display(Markdown("### 📈 Ma trận kết quả mAP học trọn đời:"))
    display(df)
    
    # Tính Forgetting & Plasticity cuối cùng sau Task 4
    if "task_4" in history and "task_1" in history:
        ap_t1_t1 = history["task_1"]["T1"]["mAP"]
        ap_t1_t4 = history["task_4"]["T1"]["mAP"]
        ap_t2_t2 = history["task_2"]["T2"]["mAP"]
        ap_t2_t4 = history["task_4"]["T2"]["mAP"]
        ap_t3_t3 = history["task_3"]["T3"]["mAP"]
        ap_t3_t4 = history["task_4"]["T3"]["mAP"]
        
        f1 = max(0.0, ap_t1_t1 - ap_t1_t4)
        f2 = max(0.0, ap_t2_t2 - ap_t2_t4)
        f3 = max(0.0, ap_t3_t3 - ap_t3_t4)
        forgetting = (f1 + f2 + f3) / 3.0
        plasticity = history["task_4"]["T4"]["mAP"]
        overall = plasticity - forgetting
        
        summary_md = f"""
### 📊 Chỉ số học trọn đời tích hợp (sau Task 4):
*   **Plasticity (Khả năng tiếp thu mới):** `{plasticity:.4f}`
*   **Forgetting (Độ quên lãng trung bình):** `{forgetting:.4f} ({forgetting*100:.2f}%)`
*   **Overall Change (Độ ổn định hệ thống):** `{overall:.4f}`
"""
        display(Markdown(summary_md))
else:
    print("-> File history_metrics.json không tồn tại. Hãy chạy đầy đủ các tác vụ huấn luyện và đánh giá trước.")